# 04 — Classical ML for Motor Imagery Classification

We have cleaned epochs from Chunk 2 and feature extractors from Chunk 4's
module additions. This notebook fits classical ML pipelines on three
feature families and compares them honestly.

**Feature families:**
1. Motor-channel log band power (18 features) — interpretable baseline
2. Lateralization indices (6 features) — informed by Chunk 3's asymmetric
   lateralization finding
3. CSP log-variance (6 features) — MI-BCI standard

**Pre-registered prediction (before any model runs):**
- Single-subject accuracy: **65–75%**
- Rationale: Chunk 3 showed mu ERD on 3/4 sensorimotor panels (moderate
  responder), with asymmetric lateralization (T2 clean, T1 bilateral).
  Strong responders typically hit 80%+; non-responders ~chance; this
  subject's profile suggests moderate decoding performance.
- CSP expected to slightly edge band power; lateralization indices
  uncertain (the feature directly encodes our finding, but only 6
  features might underspecify the decision boundary).

**Methodology:**
- 5-fold stratified CV × 10 repeats = 50 accuracy estimates per pipeline
- All learned components (CSP, scaler) refit inside CV folds
- Chance-level baselines: theoretical (50%) and empirical binomial
  upper-bound (~64%)
- Statistical comparison between pipelines via paired t-test on
  repeated-CV scores

**What this notebook does NOT do** (parked for v2 or later chunks):
- Hyperparameter tuning (nested CV) — single-subject N too small to
  spend folds on
- Riemannian features
- Cross-subject generalization (Chunk 5)
- Deep learning (Chunk 5)

## Data Preparation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from pathlib import Path
from scipy import stats

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    cross_val_score,
    cross_val_predict,
)
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    roc_auc_score,
)

import sys

# Make src importable from the notebooks/ directory
sys.path.insert(0, str(Path.cwd().parent / "src"))

import features as feat
import visualization as viz  

mne.set_log_level("WARNING")

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../data/models")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 5
N_REPEATS = 10

SyntaxError: from __future__ imports must occur at the beginning of the file (features.py, line 349)

**Load cleaned epochs**

In [ ]:
epochs = mne.read_epochs(DATA_DIR / "subject_001-epo.fif", preload=True)
y = epochs.events[:, -1].astype(np.int64)

print(epochs)
print(f"Class distribution: "
      f"{[(int(c), int(np.sum(y == c))) for c in np.unique(y)]}")
print(f"Total epochs: {len(epochs)}")
print(f"Time range: {epochs.tmin:.2f} to {epochs.tmax:.2f} s")

**Chance-level baselines**

Two relevant chance levels for honest interpretation:

1. **Theoretical chance:** 50% for balanced binary classification.
2. **Empirical "lucky chance":** the upper bound of the binomial
   confidence interval at N trials, α=0.05. With N=44, even a
   purely-guessing classifier can achieve ~64% accuracy by chance
   on a single test split. This is why we use repeated CV — averaging
   50 estimates collapses this variance.

A reported accuracy needs to clear *both* bars to mean something, and
ideally clear them with a confidence interval that doesn't overlap
"lucky chance."

In [ ]:
n_trials = len(epochs)
# Empirical lucky-chance upper bound: binomial CI upper at p=0.5, α=0.05
lucky_chance_upper = stats.binom.ppf(0.975, n_trials, 0.5) / n_trials

print(f"Theoretical chance: 0.500")
print(f"Empirical 'lucky chance' upper (binomial 95% CI, N={n_trials}): "
      f"{lucky_chance_upper:.3f}")
print(f"=> Accuracies below {lucky_chance_upper:.3f} are not distinguishable "
      f"from chance on a single split.")

**Cross-validation scaffold**

We define one CV splitter and reuse it across all pipelines. Reusing
the same random_state means each pipeline is evaluated on identical
train/test splits, which makes per-fold comparison meaningful (paired
t-test possible at the end).

**Why repeated CV instead of single 5-fold:** With ~9 test trials per
fold, single-fold accuracy has ~±15% std error. 50 estimates (5 folds
× 10 repeats) reduces this to ~±2%, which is where the comparison
between pipelines becomes resolvable.

In [ ]:
cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)

# Helper for consistent reporting
def report_cv(name, scores):
    print(f"{name:30s}  mean={scores.mean():.3f}  "
          f"std={scores.std():.3f}  "
          f"95% CI=[{np.percentile(scores, 2.5):.3f}, "
          f"{np.percentile(scores, 97.5):.3f}]  "
          f"n_above_lucky={int(np.sum(scores > lucky_chance_upper))}/"
          f"{len(scores)}")

## CSP Feature Extraction

*To be implemented in upcoming chunk.*

## Pipeline 1: Motor-channel band power + LDA

**Why LDA:** With 18 features and ~35 training epochs per fold (44 × 4/5),
we're in the regime where LDA's strong Gaussian assumption is helpful,
not hurtful. LDA on log-band-power is the original Pfurtscheller setup
and remains a hard-to-beat baseline.

**Why no hyperparameter tuning:** LDA with shrinkage has one knob
(`shrinkage='auto'` uses Ledoit-Wolf analytic shrinkage). At this N,
nested CV for tuning costs more than it gains.

Note that `StandardScaler` precedes LDA. LDA technically doesn't need
scaled features, but with shrinkage it helps the regularization act
uniformly across features.

In [ ]:
# Stateless features — safe to compute once outside CV
X_bp, bp_names, _ = feat.compute_motor_bandpower(epochs)
print(f"Band power feature matrix: {X_bp.shape}")
print(f"First 4 feature names: {bp_names[:4]}")

pipe_bp = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
])

scores_bp = cross_val_score(pipe_bp, X_bp, y, cv=cv, scoring="accuracy", n_jobs=-1)
report_cv("Band power + LDA", scores_bp)

## Pipeline 2: Lateralization indices + Logistic Regression

The "informed by Chunk 3" pipeline. 6 features (3 pairs × 2 bands)
directly encoding the C3-vs-C4 power asymmetry that the TFR analysis
showed is class-discriminative.

**Why LogReg instead of LDA:** With only 6 features, the LDA Gaussian
assumption matters less (small feature space, well-conditioned
covariance). LogReg with L2 regularization is a more transparent fit:
each coefficient maps to "how much this lateralization index moves
the decision."

**Hypothesis to test:** Given Chunk 3's finding that T2 lateralizes
cleanly and T1 is bilateral, we expect the C3-C4 mu LI coefficient
to be the largest. We'll inspect coefficients after fitting on full
data (for interpretation only — accuracy still comes from CV).

In [ ]:
X_li, li_names, li_pairs_used = feat.compute_lateralization_indices(epochs)
print(f"Lateralization feature matrix: {X_li.shape}")
print(f"Feature names: {li_names}")

pipe_li = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(penalty="l2", C=1.0,
                               solver="liblinear",
                               random_state=RANDOM_STATE)),
])

scores_li = cross_val_score(pipe_li, X_li, y, cv=cv, scoring="accuracy", n_jobs=-1)
report_cv("Lateralization + LogReg", scores_li)

In [ ]:
# Fit on full data for COEFFICIENT INSPECTION ONLY (not for accuracy)
pipe_li.fit(X_li, y)
coefs = pipe_li.named_steps["clf"].coef_.flatten()

fig, ax = plt.subplots(figsize=(8, 4))
order = np.argsort(np.abs(coefs))[::-1]
ax.barh([li_names[i] for i in order], coefs[order])
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("LogReg coefficient (standardized features)")
ax.set_title("Lateralization feature importance\n(fit on full data — interpretation only)")
plt.tight_layout()
plt.show()

## Pipeline 3: CSP + LDA

The literature-standard MI pipeline. CSP learns spatial filters that
maximize variance ratio between classes; LDA classifies on the resulting
log-variance features.

**Critical leakage point:** CSP is fit on labels, so it MUST be inside
the CV pipeline. Fitting CSP on the full data first and then doing
CV on the resulting features would leak test labels into the spatial
filters. The sklearn `Pipeline` handles this — `cross_val_score` will
refit the whole pipeline per fold.

**A note on input shape:** unlike band-power and lateralization features,
CSP needs 3D input `(n_epochs, n_channels, n_times)`. We pass the raw
epoch tensor cropped to the imagery window.

In [ ]:
X_csp_input, y_csp, ch_names = feat.epochs_to_array(epochs, tmin=0.5, tmax=3.5)
print(f"CSP input shape: {X_csp_input.shape}  (epochs, channels, times)")

# CSP doesn't need StandardScaler — log-variance output is already
# on a reasonable scale, and scaling would mix info across CSP components
# in a way that's hard to interpret.
pipe_csp = Pipeline([
    ("csp", feat.CSPFeatures(n_components=6, reg="ledoit_wolf")),
    ("clf", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
])

scores_csp = cross_val_score(pipe_csp, X_csp_input, y_csp, cv=cv,
                             scoring="accuracy", n_jobs=-1)
report_cv("CSP + LDA", scores_csp)

In [ ]:
# For visualization only — fit on full data. NOT used for accuracy.
csp_full = feat.CSPFeatures(n_components=6, reg="ledoit_wolf")
csp_full.fit(X_csp_input, y_csp)

# Build an info object matching the CSP input channels
info_csp = epochs.copy().crop(0.5, 3.5).info

fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
for i, ax in enumerate(axes):
    mne.viz.plot_topomap(csp_full.patterns_[i], info_csp, axes=ax,
                          show=False, cmap="RdBu_r")
    ax.set_title(f"CSP{i+1}")
plt.suptitle("CSP spatial patterns (full-data fit, for visualization only)", y=1.05)
plt.tight_layout()
plt.show()

## Pipeline 4: CSP + Linear SVM

Same CSP features, different classifier. LDA and SVM make different
modeling assumptions:

- **LDA** assumes Gaussian class-conditional distributions with shared
  covariance, fits a linear boundary via likelihood maximization.
- **Linear SVM** makes no distributional assumption, fits a linear
  boundary by margin maximization with hinge loss.

For features that are well-approximated as Gaussian (log-variance often
is), LDA is theoretically optimal. SVM tends to be more robust when
that assumption is shaky. Running both on identical CSP features tests
which assumption fits this subject's data better.

**Why linear SVM, not RBF:** RBF SVM adds `gamma` as a hyperparameter
on top of `C`. With N=44 and no nested CV budget, picking RBF defaults
is a coin flip. Linear SVM has one knob (`C`), defaults are reasonable,
and the comparison to LDA stays apples-to-apples on the same feature
space. RBF parked for v2.

In [ ]:
from sklearn.svm import LinearSVC

pipe_csp_svm = Pipeline([
    ("csp", feat.CSPFeatures(n_components=6, reg="ledoit_wolf")),
    ("scaler", StandardScaler()),  # SVM does need scaled features
    ("clf", LinearSVC(C=1.0, max_iter=5000, random_state=RANDOM_STATE)),
])

scores_csp_svm = cross_val_score(pipe_csp_svm, X_csp_input, y_csp,
                                  cv=cv, scoring="accuracy", n_jobs=-1)
report_cv("CSP + Linear SVM", scores_csp_svm)

### Classifier comparison on identical features

CSP features, two classifiers, paired CV folds. A direct test of
inductive-bias fit.

In [ ]:
t, p = stats.ttest_rel(scores_csp, scores_csp_svm)
print(f"CSP + LDA:        {scores_csp.mean():.3f} ± {scores_csp.std():.3f}")
print(f"CSP + Linear SVM: {scores_csp_svm.mean():.3f} ± {scores_csp_svm.std():.3f}")
print(f"Paired difference: {scores_csp.mean() - scores_csp_svm.mean():+.3f}  p={p:.4f}")

## Combined features: do they help?

The three feature families capture overlapping but not identical
information. Band power and lateralization both derive from the same
PSD; CSP is a learned spatial decomposition. We test two combinations:

- **Band power + Lateralization** (24 stateless features) — pure
  spectral features, can compute outside CV.
- **All three** — requires CSP-in-pipeline; we hstack inside the CV
  loop via a custom transformer.

**Hypothesis:** Combining helps modestly if at all. With N=44 and
already 6–18 features per family, adding more feature dimensions
risks overfitting more than it adds signal. The honest finding might
be that one family alone is best.

In [ ]:
# Stateless combination — safe outside CV
fset_combined = feat.build_ml_feature_set(
    epochs,
    feature_types=("bandpower", "lateralization"),
)
print(f"Combined feature matrix: {fset_combined.X.shape}")

pipe_combined = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")),
])

scores_combined = cross_val_score(pipe_combined, fset_combined.X, fset_combined.y,
                                   cv=cv, scoring="accuracy", n_jobs=-1)
report_cv("BandPower + Lateralization + LDA", scores_combined)

In [ ]:
# Cleaner: a custom transformer that takes a dict-like input is overkill
# for a notebook. Pragmatic approach: compute combined accuracy via
# manual CV loop. Five lines, no API gymnastics, no leakage.

from sklearn.model_selection import StratifiedKFold

X_3d = X_csp_input          # for CSP
X_2d = fset_combined.X       # band power + lateralization, pre-computed
y_arr = y_csp                # same labels in same order

manual_scores = []
manual_cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS,
                                     random_state=RANDOM_STATE)

for train_idx, test_idx in manual_cv.split(X_3d, y_arr):
    # Fit CSP on train fold ONLY
    csp = feat.CSPFeatures(n_components=6, reg="ledoit_wolf")
    csp.fit(X_3d[train_idx], y_arr[train_idx])
    
    # Transform both folds
    csp_train = csp.transform(X_3d[train_idx])
    csp_test  = csp.transform(X_3d[test_idx])
    
    # Concatenate with stateless features (safe — stateless features
    # don't see labels, so the row split is identical to a CV split
    # on pre-computed features)
    X_train = np.hstack([csp_train, X_2d[train_idx]])
    X_test  = np.hstack([csp_test,  X_2d[test_idx]])
    
    # Scale and classify
    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_test_s  = scaler.transform(X_test)
    
    clf = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    clf.fit(X_train_s, y_arr[train_idx])
    acc = clf.score(X_test_s, y_arr[test_idx])
    manual_scores.append(acc)

scores_all = np.array(manual_scores)
report_cv("CSP + BP + Lateralization + LDA", scores_all)

## Pipeline comparison

Four pipelines, 50 CV scores each. We compare via:
1. Boxplot of accuracy distributions
2. Paired t-test on per-fold scores (same splits across pipelines,
   so paired test is appropriate and more powerful than unpaired)

In [ ]:
results = {
    "Band power + LDA": scores_bp,
    "Lateralization + LogReg": scores_li,
    "CSP + LDA": scores_csp,
    "BP + Lat + LDA": scores_combined,
    "All three + LDA": scores_all,
}

fig, ax = plt.subplots(figsize=(9, 5))
positions = range(len(results))
bp = ax.boxplot(results.values(), positions=positions, widths=0.6,
                patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("#88aacc")
    patch.set_alpha(0.7)
ax.axhline(0.5, color="gray", linestyle="--", label="Chance (0.5)")
ax.axhline(lucky_chance_upper, color="red", linestyle=":",
           label=f"Lucky chance upper ({lucky_chance_upper:.2f})")
ax.set_xticks(positions)
ax.set_xticklabels(results.keys(), rotation=20, ha="right")
ax.set_ylabel("CV accuracy")
ax.set_title(f"Pipeline comparison ({N_SPLITS}-fold × {N_REPEATS} repeats, N={n_trials})")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
print("Paired t-tests (across 50 matched CV folds):\n")
pipelines = list(results.items())
for i in range(len(pipelines)):
    for j in range(i + 1, len(pipelines)):
        name_i, scores_i = pipelines[i]
        name_j, scores_j = pipelines[j]
        t, p = stats.ttest_rel(scores_i, scores_j)
        diff = scores_i.mean() - scores_j.mean()
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"  {name_i:30s} vs {name_j:30s}  "
              f"diff={diff:+.3f}  p={p:.4f} {sig}")

## Where do classifiers make mistakes?

Accuracy hides asymmetric errors. Given Chunk 3's finding that T2 (right
hand) lateralizes cleanly while T1 (left hand) is bilateral, we predict:
**T2 should be classified more accurately than T1.** Confusion matrices
will show whether this prediction holds and is a real test of the
Chunk-3-informs-Chunk-4 narrative.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, X_in, y_in) in zip(
    axes,
    [("Band power + LDA", X_bp, y),
     ("Lateralization + LogReg", X_li, y),
     ("CSP + LDA", X_csp_input, y_csp)]
):
    pipe = {"Band power + LDA": pipe_bp,
            "Lateralization + LogReg": pipe_li,
            "CSP + LDA": pipe_csp}[name]
    y_pred = cross_val_predict(pipe, X_in, y_in,
                                cv=StratifiedKFold(N_SPLITS, shuffle=True,
                                                    random_state=RANDOM_STATE),
                                n_jobs=-1)
    cm = confusion_matrix(y_in, y_pred)
    im = ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center",
                color="white" if v > cm.max() / 2 else "black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["T1 (L)", "T2 (R)"])
    ax.set_yticklabels(["T1 (L)", "T2 (R)"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(name)
plt.tight_layout()
plt.show()

## Findings

*(Fill in after running. Template based on expected outcomes.)*

**Results vs. pre-registered prediction (65–75%):**
- _(fill in)_ Best pipeline: _____ at _____% mean CV accuracy
- _(fill in)_ Prediction held / didn't hold

**Per-pipeline observations:**
- Band power + LDA: _____
- Lateralization + LogReg: _____ (was C3-C4 mu the dominant coefficient?)
- CSP + LDA: _____ (did patterns localize to motor cortex?)
- Combined: _____ (did combining help or hurt?)

**Confusion matrix:** _(check)_ Was T2 classified more accurately than
T1, consistent with the Chunk 3 lateralization finding?

**Distance from lucky-chance upper bound:**
- All pipelines clear it: weak generalization claim possible
- Some pipelines don't: report honestly; the signal is at the edge
  of detectability at this N

## Parked followups

- Generalize to 109 subjects — within-subject CV (separate model per
  subject) and cross-subject CV via `GroupKFold(groups=subject_id)`.
  Reuse this notebook's `Pipeline` objects unchanged.
- Hyperparameter tuning via nested CV — currently we use sklearn
  defaults; CSP `n_components` ∈ {4, 6, 8} and LogReg `C` ∈ {0.1, 1, 10}
  worth sweeping when sample size supports it.
- Probability calibration — current pipelines output hard predictions;
  for a BCI dashboard we'd want calibrated probabilities (Platt scaling
  or isotonic).
- Riemannian / tangent-space features — Tier 3 stretch, parked for v2.
- Filter-bank CSP — parked for v2.
- Subject-level metadata (handedness from PhysioNet records) — would
  let us test whether T1/T2 asymmetry correlates with handedness across
  subjects (Chunk 3 followup, surfaces here too).

In [ ]:
np.savez(OUT_DIR / "subject_001_classical_ml_results.npz",
         scores_bp=scores_bp,
         scores_li=scores_li,
         scores_csp=scores_csp,
         scores_combined=scores_combined,
         scores_all=scores_all,
         n_trials=n_trials,
         n_splits=N_SPLITS,
         n_repeats=N_REPEATS,
         lucky_chance_upper=lucky_chance_upper)
print(f"Saved CV results to {OUT_DIR / 'subject_001_classical_ml_results.npz'}")

## Classical ML: Logistic Regression

*To be implemented in upcoming chunk.*

## Deep Learning: EEGNet Architecture

*To be implemented in upcoming chunk.*

## EEGNet Training

*To be implemented in upcoming chunk.*

## Cross-Subject Evaluation

*To be implemented in upcoming chunk.*

## Results Comparison Table

*To be implemented in upcoming chunk.*

## Confusion Matrices

*To be implemented in upcoming chunk.*

## Discussion

*To be implemented in upcoming chunk.*